In [1]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

open_ai_base_url = os.getenv("OPENROUTER_BASE_URL")
open_ai_api_key = os.getenv("OPENROUTER_API_KEY")

headers = {
  'Authorization': f"Bearer {open_ai_api_key}",
  'Content-Type': 'application/json',
}

response = requests.post('https://eu.openrouter.ai/api/v1/chat/completions', headers=headers, json={
  'model': 'meta-llama/llama-3.1-405b',
  'messages': [{ 'role': 'user', 'content': 'Привіт, коротко представся.' }],
})

print(response.json())

{'error': {'message': 'meta-llama/llama-3.1-405b is no longer available on OpenRouter: no provider serves this model anymore. See available models at https://openrouter.ai/models', 'code': 404}, 'user_id': 'user_3LN9GN2sEmyvpwggGAv2F0VQyDy'}


In [19]:
import requests

# 1. Fetch available models from OpenRouter API
url = "https://openrouter.ai/api/v1/models"
response = requests.get(url)

if response.status_code == 200:
    models_data = response.json().get('data', [])

    # 2. Organize models by provider and base family slug
    all_model_ids = {model['id'] for model in models_data}

    base_models = []
    instruct_models = []
    pairs = []

    # 3. Categorize models as Base or Instruct
    for model in models_data:
        m_id = model['id']
        name = model.get('name', '').lower()
        desc = model.get('description', '').lower()
        instruct_type = model.get('architecture', {}).get('instruct_type')

        # Determine if model is Instruct/Chat tuned
        is_instruct = (
            'instruct' in m_id.lower()
            or 'chat' in m_id.lower()
            or (instruct_type is not None and instruct_type != 'none')
        )

        if is_instruct:
            instruct_models.append(m_id)
        else:
            base_models.append(m_id)

    print(f"Total Models: {len(models_data)}")
    print(f"Instruct/Chat Models: {len(instruct_models)}")
    print(f"Potential Base Models: {len(base_models)}\n")

    # 4. Pair Base and Instruct variants from the same family
    for instruct in instruct_models:
        # Generate possible base model ID variants
        base_candidates = [
            instruct.replace('-instruct', ''),
            instruct.replace(':instruct', ''),
            instruct.replace('-instruct', '-base'),
            instruct.replace('-chat', ''),
            instruct.replace(':chat', '')
        ]

        for candidate in base_candidates:
            if candidate in all_model_ids and candidate != instruct:
                pairs.append((candidate, instruct))
                break

    # 5. Display paired results
    if pairs:
        print(f"Found {len(pairs)} Base & Instruct Model Pair(s):")
        for base, instruct in pairs:
            print(f"  Base:     {base}")
            print(f"  Instruct: {instruct}\n")
    else:
        print("No exact 1-to-1 Base/Instruct string-matched pairs currently active on OpenRouter.")
        print("\nSample Base Models:")
        for b in base_models[:5]:
            print(f"  {b}")
        print("\nSample Instruct Models:")
        for i in instruct_models[:5]:
            print(f"  {i}")
else:
    print(f"Failed to fetch models. Status code: {response.status_code}")

Total Models: 456
Instruct/Chat Models: 47
Potential Base Models: 409

Found 1 Base & Instruct Model Pair(s):
  Base:     openai/gpt-5.2
  Instruct: openai/gpt-5.2-chat



In [21]:
import requests

HEADERS = {
    'Authorization': f"Bearer {open_ai_api_key}",
    'Content-Type': 'application/json',
}

base_model = 'openai/gpt-5.2'
instruct_model = 'openai/gpt-5.2-chat'

prompt = "Привіт, коротко представся."

def query_model(model_name, user_prompt):
    payload = {
        'model': model_name,
        'messages': [{'role': 'user', 'content': user_prompt}]
    }
    res = requests.post('https://openrouter.ai/api/v1/chat/completions', headers=HEADERS, json=payload)
    return res.json()

print("--- Base Model Response ---")
base_res = query_model(base_model, prompt)
print(base_res.get('choices', [{}])[0].get('message', {}).get('content', base_res))

print("\n--- Instruct Model Response ---")
instruct_res = query_model(instruct_model, prompt)
print(instruct_res.get('choices', [{}])[0].get('message', {}).get('content', instruct_res))

--- Base Model Response ---
{'error': {'message': 'google/gemma-2-9b is not a valid model ID', 'code': 400}, 'user_id': 'user_3LN9GN2sEmyvpwggGAv2F0VQyDy'}

--- Instruct Model Response ---
{'error': {'message': 'google/gemma-2-9b-it is no longer available on OpenRouter: no provider serves this model anymore. See available models at https://openrouter.ai/models', 'code': 404}, 'user_id': 'user_3LN9GN2sEmyvpwggGAv2F0VQyDy'}
